# ActivityLink — training isolato e benchmark storico
Protocollo: StandardScaler + LinearSVC(C=0.1), senza ricerca di feature, soglie o iperparametri.
Cinque fold GroupKFold sul SOLO training ufficiale, separati per soggetto. Scaler riappreso dentro ogni fold.
Il test ufficiale non è letto da ml/retrain.py e non determina accettazione o scelta del modello.
La valutazione del test è una fase separata e soltanto descrittiva: il benchmark era già stato consultato in passato. Non è un nuovo test cieco.
Le scelte storiche della ricetta non vengono rese indipendenti retroattivamente dalla nuova validazione interna.
Il modello congelato dei dongle resta distinto dal candidato riaddestrato; il notebook non modifica il firmware.
Dataset: UCI HAR, DOI https://doi.org/10.24432/C54S4K, CC BY 4.0. Esecuzione locale sul Mac, non sul servizio Colab.
Aprire dalla root del progetto. Dettagli: docs/ML_VALIDATION.md.


In [1]:
from pathlib import Path
import sys, subprocess, json, hashlib, os

ROOT = Path.cwd()
assert (ROOT / 'ml/retrain.py').is_file(), 'Impostare la directory corrente sulla root ActivityLink'
import numpy as np, sklearn

assert (
    np.__version__ == '2.3.5' and sklearn.__version__ == '1.8.0'
), 'Installare requirements-ml.txt con pip e riavviare il kernel'
print(
    'Python', sys.version.split()[0], 'NumPy', np.__version__, 'scikit-learn', sklearn.__version__
)
OUT = Path(os.environ.get('ACTIVITYLINK_RESULTS_DIR', ROOT / 'results/notebook-new'))
OUT.mkdir(parents=True, exist_ok=True)
DATASET = Path(os.environ.get('ACTIVITYLINK_DATASET', ROOT / 'results/har-data/UCI HAR Dataset'))


Python 3.13.5 NumPy 2.3.5 scikit-learn 1.8.0


In [2]:
args = [sys.executable, 'ml/retrain.py', '--out', str(OUT)]
if DATASET.is_dir():
    args += ['--dataset', str(DATASET)]
else:
    DATASET = OUT / 'UCI HAR Dataset'
r = subprocess.run(args, check=True, capture_output=True, text=True)
print(r.stdout)


Fold 1/5: macro-F1=0.890867
Fold 2/5: macro-F1=0.899766
Fold 3/5: macro-F1=0.986751
Fold 4/5: macro-F1=0.956699
Fold 5/5: macro-F1=0.948408
{
  "scope": "internal grouped validation of a fixed historical recipe; not a new blind external test",
  "official_test_used": false,
  "model_selection_on_test": false,
  "recipe": {
    "features": 561,
    "C": 0.1,
    "dual": "auto",
    "max_iter": 5000,
    "random_state": 42,
    "folds": 5,
    "grouping": "subject",
    "feature_selection": false,
    "hyperparameter_search": false,
    "rejection_threshold": false
  },
  "folds": [
    {
      "fold": 1,
      "training_subjects": [
        1,
        3,
        5,
        6,
        7,
        8,
        11,
        16,
        17,
        21,
        22,
        23,
        26,
        27,
        28,
        29,
        30
      ],
      "validation_subjects": [
        14,
        15,
        19,
        25
      ],
      "subject_overlap": 0,
      "training_rows": 5932,
      "val

In [3]:
m = json.loads((ROOT / 'models/svm561.json').read_text())
print('Export del riferimento congelato: models/svm561.json; retraining candidato separato in OUT.')
assert m['classes'] == [1, 2, 3, 4, 5, 6] and m['indices'] == list(range(561))
lines = ['/* Input: all 561 UCI HAR features in original dataset order */', '#include <stddef.h>']
for name, arr in [
    ('weights', np.asarray(m['weights'], dtype=np.float32).ravel()),
    ('bias', np.asarray(m['bias'], dtype=np.float32)),
]:
    lines.append(
        f'static const float har_{name}[{arr.size}]={{'
        + ','.join(f'{float(v):.9e}f' for v in arr)
        + '};'
    )
lines.append(
    'int har_predict(const float x[561]) {\n  int best = 0;\n  float top = 0;\n  for (int c = 0; c < 6; c++) {\n    float s = har_bias[c];\n    for (int j = 0; j < 561; j++)\n      s += har_weights[c * 561 + j] * x[j];\n    if (c == 0 || s > top) {\n      top = s;\n      best = c;\n    }\n  }\n  return best + 1;\n}'
)
header = '\n'.join(lines)
(OUT / 'har_linear_predictor.h').write_text(header)
assert (
    header == (ROOT / 'firmware/node/model.h').read_text()
), 'Export diverso: indagare, non sostituire automaticamente'
print(
    'Header C rigenerato identico al firmware. SHA-256:',
    hashlib.sha256(header.encode()).hexdigest(),
)


Export del riferimento congelato: models/svm561.json; retraining candidato separato in OUT.
Header C rigenerato identico al firmware. SHA-256: 998dd3f949ff30a3fbb0564ec623e837623cc87fc46c84cfa5878c3b72da7f99


In [4]:
print('Validazione interna sul SOLO training:')
validation = json.loads((OUT / 'validation.json').read_text())
print('OOF macro-F1:', validation['oof_macro_f1'])
print(
    'Media e deviazione standard dei fold:',
    validation['fold_macro_f1_mean'],
    validation['fold_macro_f1_std'],
)
assert not validation['official_test_used']
assert all(f['subject_overlap'] == 0 for f in validation['folds'])
print('Benchmark storico separato: nessuna selezione o modifica del candidato.')
r = subprocess.run(
    [
        sys.executable,
        'ml/evaluate_historical.py',
        '--dataset',
        str(DATASET),
        '--candidate',
        str(OUT / 'svm561.json'),
        '--output',
        str(OUT / 'historical-test.json'),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(r.stdout)


Validazione interna sul SOLO training:
OOF macro-F1: 0.9366243546508001
Media e deviazione standard dei fold: 0.9364979916557573 0.04033231647562571
Benchmark storico separato: nessuna selezione o modifica del candidato.


{
  "scope": "historical benchmark only; already consulted; no training or selection",
  "source": "https://doi.org/10.24432/C54S4K",
  "test_rows": 2947,
  "test_subjects": [
    2,
    4,
    9,
    10,
    12,
    13,
    18,
    20,
    24
  ],
  "subject_overlap": 0,
  "accuracy": 0.9619952494061758,
  "macro_f1": 0.9629136765287276,
  "errors": 112,
  "confusion_matrix": [
    [
      494,
      2,
      0,
      0,
      0,
      0
    ],
    [
      12,
      459,
      0,
      0,
      0,
      0
    ],
    [
      2,
      4,
      413,
      1,
      0,
      0
    ],
    [
      0,
      4,
      0,
      426,
      61,
      0
    ],
    [
      0,
      0,
      0,
      15,
      517,
      0
    ],
    [
      0,
      0,
      0,
      0,
      11,
      526
    ]
  ],
  "classification_report": {
    "1": {
      "precision": 0.9724409448818898,
      "recall": 0.9959677419354839,
      "f1-score": 0.9840637450199203,
      "support": 496.0
    },
    "2": {
      "p

In [5]:
r = subprocess.run(
    [
        sys.executable,
        'scripts/verify_model.py',
        '--dataset',
        str(DATASET),
        '--output',
        str(OUT / 'equivalence.json'),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(r.stdout)


{
  "scope": "host GCC; shipped JSON vs shipped C; no ARM execution",
  "samples": 2947,
  "mismatches": 0,
  "header_sha256": "998dd3f949ff30a3fbb0564ec623e837623cc87fc46c84cfa5878c3b72da7f99",
  "compiler": "Apple clang version 14.0.0 (clang-1400.0.29.202)"
}



## Interpretazione
La validazione OOF valuta la ricetta fissa su soggetti esclusi a turno dal training del fold; non è tuning e non sceglie il modello.
La media dei fold e la metrica aggregata OOF hanno pesature diverse e non vanno confuse. La deviazione standard non è un intervallo di confidenza.
Il test ufficiale resta un benchmark storico. Nessuna soglia impone che le sue predizioni coincidano col riferimento per approvare il training.
Il confronto C/Python verifica solo l'esportazione del riferimento congelato, non la generalizzazione del candidato o un'esecuzione su ARM.
I 24 campioni dimostrativi sono un replay; non una stima aggiuntiva di accuratezza.
